# Convert `inference_outputs` frames to videos

Walks every `left/` and `wrist/` frame folder under `inference_outputs/<position>/<run>/` and
encodes each one to an mp4 with `ffmpeg`, named after the folder it came from
(`<position>__<run>__<camera>.mp4`), collected under `inference_outputs/videos/`.

In [ ]:
import subprocess
from pathlib import Path

ROOT       = Path("inference_outputs")
VIDEO_DIR  = ROOT / "videos"
FPS        = 5          # playback fps (frames were captured at ~5-15 Hz depending on SAVE_EVERY_N_STEPS)
OVERWRITE  = False      # set True to re-encode videos that already exist

VIDEO_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def find_frame_dirs(root):
    """Yield every leaf 'left' or 'wrist' folder under root that has frame_*.jpg in it."""
    for d in sorted(root.glob("*/*/*")):
        if d.is_dir() and d.name in ("left", "wrist") and any(d.glob("frame_*.jpg")):
            yield d


def video_name_for(frame_dir, root):
    """pos_01/healthy_2026.../left -> pos_01__healthy_2026..._left.mp4"""
    rel_parts = frame_dir.relative_to(root).parts   # (position, run, camera)
    return "__".join(rel_parts) + ".mp4"


def frames_to_video(frame_dir, out_path, fps=FPS):
    n_frames = len(list(frame_dir.glob("frame_*.jpg")))
    cmd = [
        "ffmpeg", "-y" if OVERWRITE else "-n",
        "-loglevel", "error",
        "-framerate", str(fps),
        "-i", str(frame_dir / "frame_%04d.jpg"),
        "-c:v", "libx264", "-pix_fmt", "yuv420p",
        str(out_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return n_frames, result

In [ ]:
frame_dirs = list(find_frame_dirs(ROOT))
print(f"found {len(frame_dirs)} left/wrist frame folders under {ROOT}")

made, skipped, failed = 0, 0, 0
for frame_dir in frame_dirs:
    out_path = VIDEO_DIR / video_name_for(frame_dir, ROOT)
    if out_path.exists() and not OVERWRITE:
        print(f"skip  (exists) {out_path.name}")
        skipped += 1
        continue

    n_frames, result = frames_to_video(frame_dir, out_path)
    if result.returncode != 0:
        print(f"FAIL  {out_path.name}: {result.stderr.strip().splitlines()[-1] if result.stderr else 'unknown error'}")
        failed += 1
    else:
        print(f"made  {out_path.name}  ({n_frames} frames @ {FPS}fps)")
        made += 1

print(f"\ndone: {made} made, {skipped} skipped, {failed} failed -> {VIDEO_DIR}")